# 03 — Theme B2–B4: Pipeline Discipline & The Leakage Audit

**Owner:** Eric Elikplim Sunu · **Branch:** `feature/theme-a`

## What this notebook audits

Machine learning models often achieve falsely high benchmark scores through **data leakage** —
allowing information from outside the training partition to influence feature engineering, scaling,
or validation splits. In healthcare and resource allocation, an un-audited model deploys false confidence,
allocating life-saving interventions to the wrong communities.

The grading rubric assigns **25% of the grade to pipeline integrity and leakage handling**.
Here, we build the disciplined pipeline and systematically compare it against **deliberate leaky counter-examples**:

| Leak Vector | The Broken Version | The Disciplined Fix | Mechanism |
|---|---|---|---|
| **1. Preprocessing** | Fit scaler/imputer on pooled data, then split | Fit preprocessors inside `Pipeline` strictly on train | Test statistics leak into train transform |
| **2. Target Encoding** | Encode district names with target mean before split | One-hot encode or use independent domain features | Feature directly contains the test target |
| **3. Spatial Leak** | Random row-wise split (stratifying by region does not fix it) | Hold out whole regions | Neighbouring districts sit on both sides of the split |
| **4. Temporal Gap** | 2022 survey predicting 2014–17 cases | Document temporal mismatch in datasheet | Future indicators used to explain past events |

With 50 districts, a test set holds 10, so one random split is an anecdote. Every comparison below is
repeated over 500 seeds before we call a difference real.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from src import features, io

SEED = io.RANDOM_SEED
df = io.load_district_cases()
print("District dataset loaded:", df.shape)


---
## 1. Split Timing Leak (Preprocessing)

**Rule:** Split first, fit second.
Scaling or imputing on pooled data allows the mean and standard deviation of the test partition
to bleed into the training set.

In [ ]:
FEATURES = ["mean_population", "net_coverage_pct"]
X_raw = df[FEATURES]
y = df["positive_cases"].to_numpy()
strata = df["region_code"]

# BROKEN on purpose: fit the scaler on all 50 districts, then split
scaler_leaky = StandardScaler()
X_leaky_scaled = scaler_leaky.fit_transform(X_raw)
X_tr_lk, X_te_lk, y_tr_lk, y_te_lk = train_test_split(
    X_leaky_scaled, y, test_size=0.2, random_state=SEED, stratify=strata
)
m_leaky = Ridge(alpha=1.0).fit(X_tr_lk, y_tr_lk)
rmse_preproc_leaky = np.sqrt(mean_squared_error(y_te_lk, m_leaky.predict(X_te_lk)))

# DISCIPLINED: split first with the project's one split function, then fit on train only
train_df, test_df = io.split_data(df, stratify_col="region_code", random_seed=SEED)
X_tr_cl, X_te_cl = train_df[FEATURES], test_df[FEATURES]
y_tr_cl, y_te_cl = train_df["positive_cases"].to_numpy(), test_df["positive_cases"].to_numpy()
pipe_clean = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
pipe_clean.fit(X_tr_cl, y_tr_cl)
rmse_preproc_clean = np.sqrt(mean_squared_error(y_te_cl, pipe_clean.predict(X_te_cl)))

print(f"Leaky Preprocessing Test RMSE       : {rmse_preproc_leaky:,.1f}")
print(f"Disciplined (Split First) Test RMSE : {rmse_preproc_clean:,.1f}")
print(f"Difference at this one seed         : {rmse_preproc_clean - rmse_preproc_leaky:,.1f} cases")

---
## 2. Target Encoding Leak

**Rule:** No feature may be computed using the target variable.
Target encoding replaces category names with the average outcome of that category across the entire dataset,
effectively passing the answer key directly into the model.

In [ ]:
# BROKEN on purpose: target-encode each district with its own outcome, then split without strata
df_target_leak = df.copy()
df_target_leak["target_encoded_district"] = df_target_leak.groupby("district")["positive_cases"].transform("mean")

X_tgt = df_target_leak[["target_encoded_district"]]
X_tr_tg, X_te_tg, y_tr_tg, y_te_tg = train_test_split(X_tgt, y, test_size=0.2, random_state=SEED)
m_tgt = Ridge(alpha=1e-3).fit(X_tr_tg, y_tr_tg)
r2_leaky_target = r2_score(y_te_tg, m_tgt.predict(X_te_tg))

# DISCIPLINED: Independent baseline without target encoding
r2_disciplined = r2_score(y_te_cl, pipe_clean.predict(X_te_cl))

print(f"Leaky Target Encoding Test R² : {r2_leaky_target:.4f} (Near 1.0 — Artificial Memorization)")
print(f"Disciplined Feature Test R²   : {r2_disciplined:.4f} (Honest Generalization)")

---
## 3. Spatial leakage: stratifying is not enough

**The worry.** Neighbouring districts share rainfall, ecology and transmission. If a test district's
neighbour is in training, the model has partly seen the answer, so a random split can flatter it.

**Why stratifying does not fix it.** A split stratified by region guarantees every region appears in
both training and test. That is good for representation, but it keeps neighbours on both sides of the
split, just as a random split does. The honest test of new geography is to hold a whole region out.

Three checks follow: one seed (this cell), the same comparison over 500 seeds, and held-out regions.

In [ ]:
# BROKEN on purpose: unstratified random row split
X_tr_rnd, X_te_rnd, y_tr_rnd, y_te_rnd = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, shuffle=True
)
pipe_rnd = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
pipe_rnd.fit(X_tr_rnd, y_tr_rnd)
rmse_spatial_leaky = np.sqrt(mean_squared_error(y_te_rnd, pipe_rnd.predict(X_te_rnd)))

# Region-stratified split from leak_cd04, same seed
rmse_spatial_clean = rmse_preproc_clean
gap_one_seed_pct = (rmse_spatial_clean - rmse_spatial_leaky) / rmse_spatial_clean * 100

print(f"Random split test RMSE, seed {SEED}     : {rmse_spatial_leaky:,.1f}")
print(f"Stratified split test RMSE, seed {SEED} : {rmse_spatial_clean:,.1f}")
print(f"Gap at this one seed: {gap_one_seed_pct:.1f}%. The two splits drew different test districts, "
      "so this alone cannot show leakage.")

**Check 1: repeat over 500 seeds.** Each test set holds 10 districts, so which districts land in it
moves the error a lot. We repeat the random split, the stratified split and the preprocessing
comparison for 500 seeds and compare distributions, not single draws.

In [ ]:
N_SEEDS = 500  # enough for stable medians and 5th to 95th percentiles
X_scaled_all = StandardScaler().fit_transform(X_raw)  # BROKEN on purpose: scaler sees all 50 districts


def ridge_rmse(X_tr, y_tr, X_te, y_te, scale=True):
    """Test RMSE of this notebook's Ridge model; scale=False when X is already scaled."""
    model = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]) if scale else Ridge(alpha=1.0)
    return float(np.sqrt(mean_squared_error(y_te, model.fit(X_tr, y_tr).predict(X_te))))


rows = []
for s in range(N_SEEDS):
    Xa, Xb, ya, yb = train_test_split(X_raw, y, test_size=0.2, random_state=s)  # BROKEN: unstratified
    tr, te = io.split_data(df, stratify_col="region_code", random_seed=s)  # disciplined
    La, Lb, la, lb = train_test_split(X_scaled_all, y, test_size=0.2, random_state=s, stratify=strata)  # BROKEN
    stratified = ridge_rmse(tr[FEATURES], tr["positive_cases"], te[FEATURES], te["positive_cases"])
    rows.append({"random": ridge_rmse(Xa, ya, Xb, yb), "stratified": stratified,
                 "preprocessing_gap": stratified - ridge_rmse(La, la, Lb, lb, scale=False)})
sweep = pd.DataFrame(rows)

print(f"Random split has the lower error in {(sweep.random < sweep.stratified).mean():.0%} of seeds")
print(f"Preprocessing gap: mean {sweep.preprocessing_gap.mean():,.1f} cases (sd {sweep.preprocessing_gap.std():,.1f}); "
      f"the leaky version looks better in {(sweep.preprocessing_gap > 0).mean():.0%} of seeds")
sweep.quantile([0.05, 0.5, 0.95]).round(0)

**Check 2: hold out whole regions.** Train on two regions and test on the third, with the same model
and the project's split function (`io.split_data(..., hold_out=region)`). This is the question that
matters for ranking places the model has not seen.

In [ ]:
held_out = {}
preds = pd.Series(index=df.index, dtype=float)
for region in sorted(df["region_code"].unique()):
    tr, te = io.split_data(df, stratify_col="region_code", hold_out=region)
    assert set(te["region_code"]) == {region} and len(tr) + len(te) == len(df)
    model = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
    preds[te.index] = model.fit(tr[FEATURES], tr["positive_cases"]).predict(te[FEATURES])
    held_out[region] = float(np.sqrt(mean_squared_error(te["positive_cases"], preds[te.index])))
rmse_region_out = float(np.sqrt(np.mean((preds - df["positive_cases"]) ** 2)))

for region, value in held_out.items():
    print(f"Held out region {region}: test RMSE {value:,.0f}")
print(f"Pooled over the three regions: {rmse_region_out:,.0f}, "
      f"{rmse_region_out / sweep.stratified.median():.1f} times the median stratified error")

---
## 4. The Leakage Audit Matrix

Summary table comparing all evaluated leakage vectors:

In [ ]:
audit_table = pd.DataFrame([
    {
        "Vector": "1. Split timing",
        "Disciplined": "Scaler fitted inside a Pipeline, on training rows only",
        "Leaky counter-example": "Scaler fitted on all 50 districts",
        "Measured effect": f"{rmse_preproc_clean - rmse_preproc_leaky:,.0f} cases at seed {SEED}; "
                           f"{sweep.preprocessing_gap.mean():,.0f} on average over {N_SEEDS} seeds",
        "Verdict": "Real mechanism, negligible here",
    },
    {
        "Vector": "2. Target encoding",
        "Disciplined": f"Features that do not use the outcome (R² {r2_disciplined:.2f})",
        "Leaky counter-example": f"District encoded with its own case count (R² {r2_leaky_target:.2f})",
        "Measured effect": f"+{r2_leaky_target - r2_disciplined:.2f} R²",
        "Verdict": "Leak confirmed; target encoding not used",
    },
    {
        "Vector": "3. Spatial",
        "Disciplined": f"Whole region held out (pooled RMSE {rmse_region_out:,.0f})",
        "Leaky counter-example": f"Random or stratified split (median RMSE {sweep.random.median():,.0f} / {sweep.stratified.median():,.0f})",
        "Measured effect": f"{rmse_region_out / sweep.stratified.median():.1f} times the stratified error",
        "Verdict": "Leak confirmed; the model does not generalise to unseen regions",
    },
    {
        "Vector": "4. Temporal",
        "Disciplined": "2022 coverage documented as a regional proxy",
        "Leaky counter-example": "Reading 2022 coverage as a cause of 2014-17 cases",
        "Measured effect": "Not measurable with these data",
        "Verdict": "Documented caveat",
    },
])
audit_table.to_markdown(index=False)